# 🚀 Automatisierter System- & Hardware-Check für NeMo / Automodel

Dieses Notebook prüft automatisch, ob dein Host-System alle Voraussetzungen (Festplattenspeicher, NVIDIA-Treiber, Docker & GPU-Passthrough) für den Container-Start erfüllt.

**Der Ablauf:**
Führe einfach die Zellen nacheinander aus. Wenn ein Check fehlschlägt, zeigt dir das Notebook gezielt an, was behoben werden muss.

In [1]:
# Import benötigter Bibliotheken
import subprocess
import shutil
import sys
import os
import socket
from pathlib import Path

class HardwareCheckError(Exception):
    """Eigene Exception für Hardware-/Umgebungsfehler."""
    pass

print("✅ Python-Umgebung bereit.")

✅ Python-Umgebung bereit.


### Schritt 1: Festplattenspeicher prüfen
Überprüft, ob genügend Speicherplatz (z.B. für Container-Images und Modell-Caches) frei ist.

In [2]:
def check_disk_space(required_gb=250):
    print("🔍 [1/4] Prüfe freien Festplattenspeicher...")
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024 ** 3)
    print(f"   --> Freier Speicherplatz: {free_gb:.2f} GB")
    
    if free_gb < required_gb:
        raise HardwareCheckError(
            f"Zu wenig Speicherplatz! Benötigt: mind. {required_gb} GB, Verfügbar: {free_gb:.2f} GB.\n"
            f"Tipp: Große Sprachmodelle und Container-Images benötigen viel Platz."
        )
    print("   ✅ Speicherkapazität ausreichend!\n")

try:
    check_disk_space(required_gb=250)
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}")

🔍 [1/4] Prüfe freien Festplattenspeicher...
   --> Freier Speicherplatz: 340.63 GB
   ✅ Speicherkapazität ausreichend!



### Schritt 2: NVIDIA-Treiber & GPU-Verfügbarkeit prüfen
Überprüft, ob `nvidia-smi` auf dem Host reagiert und wie viel VRAM zur Verfügung steht.

In [3]:
def check_nvidia_smi():
    print("🔍 [2/4] Prüfe Host-NVIDIA-Treiber & GPU-Verfügbarkeit...")
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True
        )
        gpus = result.stdout.strip().split("\n")
        print(f"   --> Gefundene GPUs ({len(gpus)}):")
        
        total_free_vram = 0
        for gpu in gpus:
            idx, name, mem_total, mem_free = [x.strip() for x in gpu.split(",")]
            free_vram = int(mem_free) / 1024
            total_free_vram += free_vram
            print(f"      • GPU {idx}: {name} | Freier VRAM: {free_vram:.2f} GB / {int(mem_total)/1024:.2f} GB")
            
        print(f"   --> Gesamter freier VRAM über alle GPUs: {total_free_vram:.2f} GB")
        if total_free_vram < 40:
            print("   ⚠️ WARNUNG: Der freie VRAM könnte für große LLMs knapp werden!")
        print("   ✅ Host GPU-Treiber erreichbar!\n")
        
    except FileNotFoundError:
        raise HardwareCheckError("'nvidia-smi' wurde nicht gefunden. Bitte installiere die NVIDIA-Treiber auf dem Host.")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(f"Fehler bei der Ausführung von nvidia-smi: {e.stderr}")

try:
    check_nvidia_smi()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Bitte behebe das Treiber-Problem, bevor du fortfährst.")

🔍 [2/4] Prüfe Host-NVIDIA-Treiber & GPU-Verfügbarkeit...
   --> Gefundene GPUs (1):
      • GPU 0: NVIDIA L40S | Freier VRAM: 44.39 GB / 44.99 GB
   --> Gesamter freier VRAM über alle GPUs: 44.39 GB
   ✅ Host GPU-Treiber erreichbar!



### Schritt 3: Docker-Installation & Daemon prüfen
Prüft, ob der Docker-Dienst läuft und der User die nötigen Berechtigungen hat.

In [4]:
def check_docker_installation():
    print("🔍 [3/4] Prüfe Docker-Installation & Daemon-Status...")
    try:
        subprocess.run(["docker", "info"], capture_output=True, text=True, check=True)
        print("   ✅ Docker ist installiert und der Daemon läuft!\n")
    except FileNotFoundError:
        raise HardwareCheckError("Docker CLI nicht gefunden. Ist Docker auf dem System installiert?")
    except subprocess.CalledProcessError:
        raise HardwareCheckError("Docker-Daemon läuft nicht oder der aktuelle Benutzer hat keine Rechte (sudo / docker group).")

try:
    check_docker_installation()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Starte den Docker-Service oder prüfe deine Benutzerrechte.")

🔍 [3/4] Prüfe Docker-Installation & Daemon-Status...
   ✅ Docker ist installiert und der Daemon läuft!



### Schritt 4: NVIDIA Container Toolkit prüfen (GPU-Passthrough)
Testet, ob ein Docker-Container erfolgreich auf die GPU zugreifen kann.

In [5]:
def check_nvidia_container_toolkit():
    print("🔍 [4/4] Prüfe NVIDIA Container Toolkit (GPU-Passthrough in Docker)...")
    cmd = [
        "docker", "run", "--rm", "--gpus", "all",
        "nvidia/cuda:12.0.0-base-ubuntu22.04",
        "nvidia-smi"
    ]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("   ✅ GPU-Durchreichung an Docker-Container funktioniert einwandfrei!\n")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(
            f"Docker kann nicht auf die GPUs zugreifen! Fehler:\n{e.stderr}\n"
            "Tipp: Stelle sicher, dass das 'nvidia-container-toolkit' installiert ist und Docker neu gestartet wurde."
        )

try:
    check_nvidia_container_toolkit()
except HardwareCheckError as e:
    print(f"❌ FEHLER: {e}\n👉 Installiere das NVIDIA Container Toolkit (siehe README).")

🔍 [4/4] Prüfe NVIDIA Container Toolkit (GPU-Passthrough in Docker)...
   ✅ GPU-Durchreichung an Docker-Container funktioniert einwandfrei!



### Schritt 5: Projekt-Verzeichnisse & Umgebung vorbereiten
Wenn alle oberen Checks erfolgreich waren, richten wir die Ordner und optional das `.env`-Token ein.

In [14]:
from pathlib import Path
import os
from dotenv import load_dotenv

def setup_project_environment():
    print("=== Bereite Projekt-Umgebung vor ===")
    dirs = [
        "/data/nemo-fraud-detection/notebooks/01_Data_Generation/data", 
        "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft"
    ]
    
    for d in dirs:
        path = Path(d)
        if path.exists():
            print(f"ℹ️ Ordner existiert bereits: {d}/")
        else:
            path.mkdir(parents=True, exist_ok=True)
            print(f"📁 Ordner neu erstellt: {d}/")
            
    print()
    
    # .env einlesen, falls vorhanden
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(env_path)
        print("ℹ️ .env Datei existiert bereits und wurde geladen.")
    
    # Aktuelle Werte aus den Umgebungsvariablen prüfen
    current_hf = os.getenv("HF_TOKEN", "").strip()
    current_ngc = os.getenv("NGC_API_KEY", "").strip()
    
    # Wenn einer der Schlüssel fehlt, nachfragen
    needs_input = not current_hf or not current_ngc
    
    if needs_input:
        print("=== API-Schlüssel Konfiguration ===")
        if not current_hf:
            hf_input = input("Gib deinen Hugging Face Token ein (optional, Enter zum Überspringen): ").strip()
            if hf_input:
                current_hf = hf_input
                os.environ["HF_TOKEN"] = current_hf
        else:
            print("ℹ️ HF_TOKEN ist bereits vorhanden.")
            
        if not current_ngc:
            ngc_input = input("Gib deinen NGC API-Key ein (optional, Enter zum Überspringen): ").strip()
            if ngc_input:
                current_ngc = ngc_input
                os.environ["NGC_API_KEY"] = current_ngc
        else:
            print("ℹ️ NGC_API_KEY ist bereits vorhanden.")
            
        # .env Datei mit dem kompletten Stand (neu oder aktualisiert) schreiben
        env_content = []
        if current_hf:
            env_content.append(f"HF_TOKEN={current_hf}")
        if current_ngc:
            env_content.append(f"NGC_API_KEY={current_ngc}")
            
        if env_content:
            env_path.write_text("\n".join(env_content) + "\n")
            print("✅ .env Datei erfolgreich aktualisiert/gespeichert.")
    else:
        print("ℹ️ Alle erforderlichen API-Schlüssel sind bereits in der Umgebung gesetzt.")

setup_project_environment()

=== Bereite Projekt-Umgebung vor ===
ℹ️ Ordner existiert bereits: /data/nemo-fraud-detection/notebooks/01_Data_Generation/data/
ℹ️ Ordner existiert bereits: /data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/

ℹ️ .env Datei existiert bereits und wurde geladen.
=== API-Schlüssel Konfiguration ===
ℹ️ HF_TOKEN ist bereits vorhanden.


Gib deinen NGC API-Key ein (optional, Enter zum Überspringen):  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


✅ .env Datei erfolgreich aktualisiert/gespeichert.


### Schritt 6: Container starten & Zugriffs-Anleitung anzeigen
Startet den gesamten Stack im Hintergrund und ermittelt automatisch die Server-IP sowie die Befehle für den Zugriff.

In [15]:
import socket
import subprocess
from pathlib import Path

def get_host_ip():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except Exception:
        return "localhost"

def start_stack_and_show_guide():
    print("=== 1. Aktive Docker-Container prüfen ===")
    try:
        ps_res = subprocess.run(["docker", "ps"], capture_output=True, text=True, check=True)
        if ps_res.stdout.strip():
            print(ps_res.stdout)
        else:
            print("ℹ️ Keine aktiven Docker-Container gefunden.\n")
    except Exception as e:
        print(f"⚠️ Konnte Docker-Status nicht abrufen: {e}\n")

    print("=== 2. Daten-Verzeichnis prüfen ===")
    data_dir = Path("/data/nemo-fraud-detection/notebooks/01_Data_Generation/data")
    if data_dir.exists():
        print(f"✅ Das Verzeichnis existiert bereits: {data_dir}\n")
    else:
        print(f"⚠️ Warnung: Das Verzeichnis existiert noch nicht: {data_dir}\n")

    print("=== 3. Starte Docker Compose Stack ===")
    try:
        res = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True, check=True)
        print("🚀 Alle Container erfolgreich gestartet!\n")
    except subprocess.CalledProcessError as e:
        print(f"❌ Fehler beim Starten des Docker-Stacks:\n{e.stderr}\n")
        return

    server_ip = get_host_ip()

    print("=" * 65)
    print("🌐 ZUGRIFFS- UND LÄUFIGKEITS-GUIDE")
    print("=" * 65)

    print("\n1. 🐳 NeMo-Container:")
    print("   - Zugriff per Terminal (SSH/Host):")
    print("     👉 docker exec -it nemo-24.09 bash")
    print("   - Jupyter Notebook darin starten (falls benötigt):")
    print("     👉 jupyter notebook --ip=0.0.0.0 --port=8889 --no-browser --allow-root")
    print("   - Funktionsprüfung: Öffne die URL im Browser. Das Jupyter-Interface sollte laden.")

    print("\n2. 🌸 NeMo Automodel Container:")
    print("   - Zugriff per Terminal (SSH/Host):")
    print("     👉 docker exec -it nemo-automodel bash")
    print("   - Jupyter Notebook darin starten (falls benötigt):")
    print("     👉 jupyter notebook --ip=0.0.0.0 --port=8887 --no-browser --allow-root")
    print(f"   - Web-Adresse: http://{server_ip}:8887 (oder http://localhost:8887)")
    print("   - Funktionsprüfung: Öffne die URL im Browser. Das Jupyter-Interface sollte laden.")

    print("\n3. 📈 Grafana Dashboard (Monitoring):")
    print(f"   - Web-Adresse: http://{server_ip}:3000")
    print("   - Standard-Login: admin / admin")
    print("   - Funktionsprüfung: Prüfe, ob das NVIDIA DCGM-Exporter Dashboard Daten (GPU-Auslastung/VRAM) anzeigt.")

    print("\n=== 3. Aktuelle Docker-Container im System ===")
    try:
        # Zeigt alle Container an (auch gestoppte, damit Prometheus, dcgm-exporter etc. sichtbar sind)
        res = subprocess.run(["docker", "ps", "-a"], capture_output=True, text=True, check=True)
        if res.stdout.strip():
            print(res.stdout)
        else:
            print("ℹ️ Keine Docker-Container gefunden.")
    except Exception as e:
        print(f"⚠️ Konnte Docker-Status nicht abrufen: {e}")

start_stack_and_show_guide()

=== 1. Aktive Docker-Container prüfen ===
CONTAINER ID   IMAGE                                     COMMAND                  CREATED          STATUS          PORTS                                                             NAMES
0027f9a65e4f   nvcr.io/nvidia/nemo-automodel:26.06.00    "/opt/nvidia/nvidia_…"   12 minutes ago   Up 12 minutes   6006/tcp, 8888/tcp, 0.0.0.0:8887->8887/tcp, [::]:8887->8887/tcp   nemo-fraud-detection-nemo-automodel-1
3aed3a282f15   nemo-fraud-detection-nemo                 "/opt/nvidia/nvidia_…"   16 minutes ago   Up 16 minutes   6006/tcp, 8888/tcp, 0.0.0.0:8889->8889/tcp, [::]:8889->8889/tcp   nemo-24.09
0dd4014918d2   nvcr.io/nvidia/k8s/dcgm-exporter:latest   "/usr/bin/dcgm-expor…"   16 minutes ago   Up 16 minutes   0.0.0.0:9400->9400/tcp, [::]:9400->9400/tcp                       dcgm-exporter
451bc7031da2   prom/prometheus:latest                    "/bin/prometheus --c…"   16 minutes ago   Up 16 minutes   0.0.0.0:9090->9090/tcp, [::]:9090->9090/tcp       

In [16]:
import socket
import urllib.request
import json

def check_container_connectivity():
    print("\n=== 4. Netzwerk- & Kommunikations-Checks ===")
    
    # Zu testende Endpunkte innerhalb des Docker-Netzwerks (oder via localhost, je nachdem wo das Skript läuft)
    # Wenn das Skript direkt auf dem Host läuft, nutzen wir localhost und die gemappten Ports:
    services_to_check = [
        ("NeMo Automodel (Jupyter)", "http://localhost:8887"),
        ("NIM Llama 3 LLM", "http://localhost:8800/v1/models"), # Typischer OpenAI-kompatibler Endpunkt von NIM
        ("Prometheus Monitoring", "http://localhost:9090"),
        ("Grafana Dashboard", "http://localhost:3000"),
        ("DCGM Exporter (GPU)", "http://localhost:9400/metrics")
    ]
    
    for name, url in services_to_check:
        try:
            # Kurzer HTTP-Request, um zu sehen, ob der Dienst antwortet
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=3) as response:
                print(f"✅ {name} ist erreichbar (Status: {response.getcode()})")
        except Exception as e:
            print(f"⚠️ {name} antwortet noch nicht (möglicherweise startet er noch): {e}")

    # Optional: Testen, ob der NIM-Container spezifisch antwortet
    print("\n--- Teste Verbindung zum LLM (NIM) ---")
    try:
        url = "http://localhost:8800/v1/models"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            print(f"🚀 LLM-Container erfolgreich angesprochen! Verfügbare Modelle: {[m['id'] for m in data.get('data', [])]}")
    except Exception:
        print("ℹ️ NIM-Container lädt das Modell im Hintergrund (das kann beim ersten Start einige Minuten dauern).")

# Am Ende deiner setup_project_environment() aufrufen:
check_container_connectivity()


=== 4. Netzwerk- & Kommunikations-Checks ===
✅ NeMo Automodel (Jupyter) ist erreichbar (Status: 200)
✅ NIM Llama 3 LLM ist erreichbar (Status: 200)
✅ Prometheus Monitoring ist erreichbar (Status: 200)
✅ Grafana Dashboard ist erreichbar (Status: 200)
✅ DCGM Exporter (GPU) ist erreichbar (Status: 200)

--- Teste Verbindung zum LLM (NIM) ---
🚀 LLM-Container erfolgreich angesprochen! Verfügbare Modelle: ['meta/llama-3.1-8b-instruct']
